# Day 4: Joint temporal detection and lineage reconstruction

This notebook trains, validates and submits one pipeline. It combines a compact temporal 3D U-Net with multiscale LoG proposals, learns backward motion and division evidence from sparse tracks, compares alternative association rules on held-out embryos, and writes `submission.csv` using only the selected configuration.


## Scientific basis

The design follows the competition [reference implementation](https://github.com/royerlab/kaggle-cell-tracking-competition): sparse detection supervision, temporal 3D features and learned association. Joint center and motion prediction follows [Malin-Mayor et al.](https://www.nature.com/articles/s41587-022-01427-7) and [MPM](https://openaccess.thecvf.com/content_CVPR_2020/html/Hayashida_MPM_Joint_Representation_of_Motion_and_Position_Map_for_Cell_CVPR_2020_paper.html). Local candidate restriction and lightweight association follow the efficiency principle demonstrated by [CELLECT](https://www.nature.com/articles/s41592-025-02886-x).

Sparse labels are never treated as a complete catalogue of cells. Unlabelled voxels receive only a weak loss. Optional fusion and division edges are retained only when they improve held-out scoring.


## 1. Environment and configuration


In [ ]:
from pathlib import Path
from itertools import product
from collections import defaultdict, OrderedDict
import json, math, time, warnings
import blosc2
import zstandard as zstd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, gaussian_laplace, maximum_filter
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
np.random.seed(42); torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| torch:', torch.__version__, '| numpy:', np.__version__)


In [ ]:
COMPETITION_SLUG='biohub-cell-tracking-during-development'
SPACING_UM=np.array([1.625,0.40625,0.40625],np.float32)
MATCH_RADIUS_UM=7.0
CFG={
 'patchShape':(24,96,96), 'trainSamples':240, 'epochs':3, 'batchSize':2, 'learningRate':2e-3,
 'logScalesUm':[0.9,1.3,1.8], 'backgroundSigmaUm':4.5, 'proposalQuantile':0.985,
 'nmsUm':1.8, 'maxProposalsPerFrame':2200, 'countMultipliers':[0.95,1.0,1.05],
 'fusionWeights':[0.0,0.30,1.0], 'motionWeights':[0.0,1.0], 'candidateK':6, 'linkGateUm':11.0,
 'divisionThresholds':[1.1,0.88], 'tilesPerBatch':4
}
print(json.dumps(CFG,indent=2))


## 2. Dependency-free Zarr v3 and GEFF readers


In [ ]:
def _dtype_from_metadata(meta):
    dtype = np.dtype(meta['data_type'])
    byte_codec = next((c for c in meta.get('codecs', []) if c.get('name') == 'bytes'), None)
    if byte_codec and dtype.itemsize > 1:
        endian = byte_codec.get('configuration', {}).get('endian', 'little')
        dtype = dtype.newbyteorder('<' if endian == 'little' else '>')
    return dtype

def decode_zarr_chunk(raw, meta, expected_nbytes=None):
    """Reverse the compression codecs declared in Zarr v3 metadata."""
    decoded = raw
    for codec in reversed(meta.get('codecs', [])):
        name = codec.get('name')
        if name == 'blosc':
            decoded = blosc2.decompress(decoded)
        elif name == 'zstd':
            decoded = zstd.ZstdDecompressor().decompress(decoded, max_output_size=expected_nbytes or 0)
        elif name == 'bytes':
            pass  # dtype and byte order are applied by np.frombuffer below
        elif name in {'crc32c'}:
            decoded = decoded[:-4]  # checksum bytes follow the encoded payload
        else:
            raise NotImplementedError(f'Unsupported Zarr codec: {name}')
    return decoded

def read_zarr_v3_array(array_path):
    array_path = Path(array_path)
    meta = json.loads((array_path / 'zarr.json').read_text())
    shape = tuple(meta['shape'])
    chunks = tuple(meta['chunk_grid']['configuration']['chunk_shape'])
    dtype = _dtype_from_metadata(meta)
    fill = meta.get('fill_value', 0)
    out = np.full(shape, 0 if fill is None else fill, dtype=dtype)
    grid = tuple(math.ceil(s / c) for s, c in zip(shape, chunks))
    for index in product(*(range(n) for n in grid)):
        path = array_path / 'c'
        for i in index:
            path /= str(i)
        if not path.exists():
            continue
        target = tuple(slice(i*c, min((i+1)*c, s)) for i, c, s in zip(index, chunks, shape))
        actual_shape = tuple(sl.stop-sl.start for sl in target)
        expected_nbytes = math.prod(chunks) * dtype.itemsize
        flat = np.frombuffer(decode_zarr_chunk(path.read_bytes(), meta, expected_nbytes), dtype=dtype)
        if flat.size == math.prod(chunks):
            chunk = flat.reshape(chunks)[tuple(slice(0,n) for n in actual_shape)]
        elif flat.size == math.prod(actual_shape):
            chunk = flat.reshape(actual_shape)
        else:
            raise ValueError(f'Unexpected decoded chunk size at {path}: {flat.size}')
        out[target] = chunk
    return out

class TimeChunkedZarr:
    def __init__(self, path):
        self.path = Path(path) / '0'
        self.metadata = json.loads((self.path / 'zarr.json').read_text())
        self.shape = tuple(self.metadata['shape'])
        self.chunks = tuple(self.metadata['chunk_grid']['configuration']['chunk_shape'])
        self.dtype = _dtype_from_metadata(self.metadata)
        if self.chunks != (1,) + self.shape[1:]:
            raise ValueError(f'Expected one full frame per chunk, found {self.chunks}')

    def __len__(self):
        return self.shape[0]

    def __getitem__(self, t):
        t = int(t) % self.shape[0]
        path = self.path / 'c' / str(t) / '0' / '0' / '0'
        raw = decode_zarr_chunk(path.read_bytes(), self.metadata, math.prod(self.chunks)*self.dtype.itemsize)
        return np.frombuffer(raw, dtype=self.dtype).reshape(self.chunks)[0]

def find_competition_root():
    candidates = [Path('/kaggle/input/competitions') / COMPETITION_SLUG, Path('/kaggle/input') / COMPETITION_SLUG]
    for path in candidates:
        if (path / 'train').exists() and (path / 'test').exists():
            return path
    raise FileNotFoundError('Attach the official competition data.')

ROOT = find_competition_root()
TRAIN_DIR, TEST_DIR = ROOT / 'train', ROOT / 'test'
train_movies = sorted(TRAIN_DIR.glob('*.zarr'))
test_movies = sorted(TEST_DIR.glob('*.zarr'))
print(ROOT, '| train:', len(train_movies), '| test:', len(test_movies))


In [ ]:
def _first_existing(base, relative_paths):
    for rel in relative_paths:
        path = base / rel
        if (path / 'zarr.json').exists():
            return path
    raise FileNotFoundError(f'None of {relative_paths} found under {base}')

def _recursive_find_key(obj, target):
    if isinstance(obj, dict):
        if target in obj:
            return obj[target]
        for value in obj.values():
            found = _recursive_find_key(value, target)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = _recursive_find_key(value, target)
            if found is not None:
                return found
    return None

def load_geff(geff_path):
    geff_path = Path(geff_path)
    ids = read_zarr_v3_array(_first_existing(geff_path, ['nodes/ids'])).reshape(-1).astype(np.int64)
    props = {}
    for key in ['t','z','y','x']:
        props[key] = read_zarr_v3_array(_first_existing(geff_path, [f'nodes/props/{key}/values', f'nodes/{key}'])).reshape(-1)
    edge_path = _first_existing(geff_path, ['edges/ids'])
    edges = read_zarr_v3_array(edge_path).reshape(-1, 2).astype(np.int64)
    nodes = pd.DataFrame({'node_id': ids, **props})
    root_meta = json.loads((geff_path / 'zarr.json').read_text())
    estimated = _recursive_find_key(root_meta, 'estimated_number_of_nodes')
    return nodes, edges, float(estimated) if estimated is not None else np.nan


probe_nodes, probe_edges, probe_estimated = load_geff(train_movies[0].with_suffix('.geff'))
print(train_movies[0].stem, len(probe_nodes), len(probe_edges), probe_estimated)


## 3. Leakage-safe movie split


In [ ]:
def embryo_id(path): return Path(path).stem.split('_')[0]
def balanced_movies(paths,n=1):
    groups=defaultdict(list)
    for path in paths: groups[embryo_id(path)].append(path)
    return [p for key in sorted(groups) for p in groups[key][:n]]

validation_movies=balanced_movies(train_movies,1)
fit_movies=[p for p in train_movies if p not in validation_movies] or train_movies
print('training movies:',len(fit_movies),'| validation:',[p.stem for p in validation_movies])


## 4. Compact temporal 3D U-Net

Three adjacent frames are input channels. The network predicts five maps: cell-center confidence, three backward-displacement components, and division confidence. Downsampling occurs only laterally, preserving the already coarse Z axis.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(nn.Conv3d(a,b,3,padding=1),nn.InstanceNorm3d(b),nn.SiLU(),
                               nn.Conv3d(b,b,3,padding=1),nn.SiLU())
    def forward(self,x): return self.net(x)

class TemporalUNet3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1=ConvBlock(3,12); self.pool=nn.MaxPool3d((1,2,2)); self.enc2=ConvBlock(12,24)
        self.up=nn.ConvTranspose3d(24,12,(1,2,2),stride=(1,2,2)); self.dec=ConvBlock(24,16)
        self.head=nn.Conv3d(16,5,1)
    def forward(self,x):
        a=self.enc1(x); b=self.enc2(self.pool(a)); return self.head(self.dec(torch.cat([a,self.up(b)],1)))

model=TemporalUNet3D().to(DEVICE)
print('trainable parameters:',sum(p.numel() for p in model.parameters()))


## 5. Sparse temporal training patches

Patches are randomly offset around annotated centers so the target does not always appear in the middle. Motion and division targets are defined only where a labeled lineage provides them.


In [ ]:
def normalize_frame(frame):
    a=np.asarray(frame,np.float32); sample=a[::max(1,a.shape[0]//32),::4,::4]
    lo,hi=np.quantile(sample,[.01,.9995]); return np.clip((a-lo)/max(float(hi-lo),1),0,1).astype(np.float32,copy=False)

def crop_pad(a,center,shape):
    center=np.asarray(center,int); shape=np.asarray(shape,int); lo=center-shape//2; hi=lo+shape
    src=tuple(slice(max(0,l),min(s,h)) for l,h,s in zip(lo,hi,a.shape))
    pad=[(max(0,-l),max(0,h-s)) for l,h,s in zip(lo,hi,a.shape)]
    return np.pad(a[src],pad,mode='reflect')

class SparsePatchDataset(Dataset):
    def __init__(self,paths,n):
        self.items=[]; self.graphs={}; self.cache=OrderedDict(); rng=np.random.default_rng(42)
        for path in paths:
            nodes,edges,_=load_geff(path.with_suffix('.geff')); self.graphs[path]=(nodes,edges)
            valid=nodes[(nodes.t>0)&(nodes.t<len(TimeChunkedZarr(path))-1)]
            take=min(len(valid),max(1,n//max(len(paths),1)))
            for i in rng.choice(len(valid),take,replace=False): self.items.append((path,valid.iloc[i]))
        rng.shuffle(self.items)
    def __len__(self): return len(self.items)
    def frame(self,path,t):
        key=(str(path),int(t))
        if key not in self.cache:
            self.cache[key]=normalize_frame(TimeChunkedZarr(path)[t])
            if len(self.cache)>8: self.cache.popitem(last=False)
        return self.cache[key]
    def __getitem__(self,i):
        path,row=self.items[i]; t=int(row.t); xyz=np.array([row.z,row.y,row.x],float)
        jitter=np.random.randint(-5,6,3); center=np.rint(xyz).astype(int)+jitter
        x=np.stack([crop_pad(self.frame(path,u),center,CFG['patchShape']) for u in [t-1,t,t+1]])
        target=np.zeros((5,*CFG['patchShape']),np.float32); target[1:4]=np.nan; q=np.asarray(CFG['patchShape'])//2-jitter
        grid=np.indices(CFG['patchShape']); dist=sum(((grid[k]-q[k])*SPACING_UM[k]/1.2)**2 for k in range(3))
        target[0]=np.exp(-dist/2)
        nodes,edges=self.graphs[path]; lookup=nodes.set_index('node_id'); uid=int(row.node_id)
        parent={int(b):int(a) for a,b in edges}; out=defaultdict(list)
        for a,b in edges: out[int(a)].append(int(b))
        positive=target[0]>.5
        if uid in parent:
            delta=(lookup.loc[parent[uid],['z','y','x']].to_numpy(float)-xyz)*SPACING_UM
            for k in range(3): target[1+k][positive]=delta[k]/8
        target[4][positive]=float(len(out[uid])==2)
        assert x.dtype==np.float32 and target.dtype==np.float32
        return torch.from_numpy(x),torch.from_numpy(target)

train_data=SparsePatchDataset(fit_movies,CFG['trainSamples'])
train_loader=DataLoader(train_data,batch_size=CFG['batchSize'],shuffle=True,num_workers=0)
print('training patches:',len(train_data))


## 6. Joint loss with hard-negative mining

Center loss is strong near annotations and weak elsewhere. The largest unexplained responses receive extra weight as hard negatives. Motion and division losses are restricted to annotated center neighborhoods.


In [ ]:
def joint_loss(pred,target):
    heat=torch.sigmoid(pred[:,0]); truth=target[:,0]; positive=truth>.1
    bce=nn.functional.binary_cross_entropy(heat,truth,reduction='none')
    weight=torch.where(positive,torch.ones_like(truth),torch.full_like(truth,.002))
    flat=heat.masked_fill(positive,-1).flatten(1); k=min(1024,flat.shape[1])
    hard=torch.zeros_like(flat); hard.scatter_(1,flat.topk(k,1).indices,1); weight+=.02*hard.view_as(weight)
    center=(bce*weight).sum()/weight.sum()
    mask=torch.isfinite(target[:,1:4])
    motion=nn.functional.smooth_l1_loss(torch.tanh(pred[:,1:4])[mask],target[:,1:4][mask]) if mask.any() else pred[:,1:4].sum()*0
    division=nn.functional.binary_cross_entropy_with_logits(pred[:,4][positive],target[:,4][positive])
    return center+.4*motion+.1*division

optimizer=torch.optim.AdamW(model.parameters(),lr=CFG['learningRate']); history=[]
for epoch in range(CFG['epochs']):
    model.train(); losses=[]; started=time.time()
    for x,y in train_loader:
        x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(set_to_none=True)
        loss=joint_loss(model(x),y); loss.backward(); optimizer.step(); losses.append(loss.item())
    row={'epoch':epoch+1,'loss':float(np.mean(losses)),'seconds':time.time()-started}; history.append(row); print(row)
pd.DataFrame(history).to_csv('/kaggle/working/trainingHistory.csv',index=False)
torch.save(model.state_dict(),'/kaggle/working/dayFourModel.pt')


## 7. Cached tiled inference and neural–LoG ensemble

The U-Net processes overlapping tiles, so memory is bounded. Each movie is inferred once. Neural-only and fused alternatives reuse the cached maps.


In [ ]:
def sigma_vox(s): return tuple(float(s)/SPACING_UM)
def odd_window(r):
    q=np.maximum(1,np.ceil(float(r)/SPACING_UM).astype(int)); return tuple(2*q+1)
def unit_scale(a):
    lo,hi=np.quantile(a,[.01,.999]); return np.clip((a-lo)/max(float(hi-lo),1e-6),0,1).astype(np.float32,copy=False)
def classical_response(frame):
    a=normalize_frame(frame); fg=np.clip(a-gaussian_filter(a,sigma_vox(CFG['backgroundSigmaUm'])),0,None)
    return unit_scale(np.maximum.reduce([-s*s*gaussian_laplace(fg,sigma_vox(s)) for s in CFG['logScalesUm']]))

def predict_maps(movie,t):
    frames=[normalize_frame(movie[min(max(u,0),len(movie)-1)]) for u in [t-1,t,t+1]]
    shape=np.array(frames[0].shape); patch=np.minimum(np.array(CFG['patchShape']),shape); stride=np.maximum(1,patch//2)
    starts=[]
    for k in range(3): starts.append(sorted(set(range(0,max(1,shape[k]-patch[k]+1),stride[k]))|{max(0,shape[k]-patch[k])}))
    out=np.zeros((5,*shape),np.float32); count=np.zeros(shape,np.float32); batch=[]; locations=[]
    def flush():
        if not batch:return
        with torch.no_grad(): pred=model(torch.from_numpy(np.stack(batch)).to(DEVICE)).cpu().numpy()
        for value,sl in zip(pred,locations): out[(slice(None),)+sl]+=value; count[sl]+=1
        batch.clear();locations.clear()
    model.eval()
    for z,y,x in product(*starts):
        sl=(slice(z,z+patch[0]),slice(y,y+patch[1]),slice(x,x+patch[2])); batch.append(np.stack([a[sl] for a in frames]).astype(np.float32,copy=False));locations.append(sl)
        if len(batch)>=CFG['tilesPerBatch']:flush()
    flush();out/=np.maximum(count,1);out[0]=torch.sigmoid(torch.from_numpy(out[0])).numpy();out[4]=torch.sigmoid(torch.from_numpy(out[4])).numpy();out[1:4]=np.tanh(out[1:4])*8
    return out

def infer_movie(path):
    movie=TimeChunkedZarr(path); result=[]
    for t in range(len(movie)):
        neural=predict_maps(movie,t); result.append((neural,classical_response(movie[t])))
        if t==0 or (t+1)%10==0:print(path.stem,t+1,'/',len(movie))
    return result


## 8. Learned association and conservative divisions

True edges are positives; each source's nearest incorrect successors are hard negatives. The final cost combines this learned geometric prior with the U-Net backward-motion residual. Hungarian assignment optimizes the complete frame pair. A second daughter is allowed only when division confidence is high and the target remains unused.


In [ ]:
def fit_edge_model(paths):
    features=[];labels=[]
    for path in paths:
        nodes,edges,_=load_geff(path.with_suffix('.geff')); lookup=nodes.set_index('node_id'); true={tuple(map(int,e)) for e in edges}
        groups={int(t):g for t,g in nodes.groupby('t')}
        for source,target in true:
            if source not in lookup.index or target not in lookup.index:continue
            origin=lookup.loc[source,['z','y','x']].to_numpy(float); delta=(lookup.loc[target,['z','y','x']].to_numpy(float)-origin)*SPACING_UM
            features.append([np.linalg.norm(delta),*np.abs(delta)]);labels.append(1)
            future=groups.get(int(lookup.loc[source,'t'])+1)
            if future is None:continue
            alternatives=(future[['z','y','x']].to_numpy(float)-origin)*SPACING_UM
            for j in np.argsort(np.linalg.norm(alternatives,axis=1))[:4]:
                uid=int(future.iloc[j].node_id)
                if (source,uid) not in true:
                    d=alternatives[j];features.append([np.linalg.norm(d),*np.abs(d)]);labels.append(0)
    return LogisticRegression(class_weight='balanced',max_iter=300).fit(features,labels)

EDGE_MODEL=fit_edge_model(fit_movies)

def extract_pool(neural,classical,fusion_weight):
    response=(1-fusion_weight)*neural[0]+fusion_weight*classical
    maxima=response==maximum_filter(response,size=odd_window(CFG['nmsUm']),mode='nearest')
    cutoff=np.quantile(response,CFG['proposalQuantile']); coords=np.argwhere(maxima&(response>=cutoff)); scores=response[tuple(coords.T)]
    if len(coords)>CFG['maxProposalsPerFrame']:
        keep=np.argpartition(scores,-CFG['maxProposalsPerFrame'])[-CFG['maxProposalsPerFrame']:];coords,scores=coords[keep],scores[keep]
    order=np.argsort(scores)[::-1];coords,scores=coords[order].astype(float),scores[order]
    motion=neural[1:4,tuple(coords.astype(int).T)].T if len(coords) else np.empty((0,3)); division=neural[4][tuple(coords.astype(int).T)] if len(coords) else np.empty(0)
    return coords,scores,motion,division

def allocate_counts(capacity,target):
    capacity=np.asarray(capacity,int);target=min(int(round(target)),int(capacity.sum()));ideal=target*capacity/max(capacity.sum(),1);counts=np.minimum(np.floor(ideal).astype(int),capacity)
    while counts.sum()<target:
        possible=np.flatnonzero(counts<capacity);j=possible[np.argmax((ideal-counts)[possible])];counts[j]+=1
    return counts

def link_pair(a,aid,b,bid,target_motion,ascore,bscore,division,threshold,motion_weight):
    if not len(a) or not len(b):return []
    delta=(b[None]-a[:,None])*SPACING_UM; raw=np.linalg.norm(delta,axis=2)
    residual=np.linalg.norm(-delta-target_motion[None,:,:],axis=2)
    probability=EDGE_MODEL.predict_proba(np.c_[raw.ravel(),np.abs(delta).reshape(-1,3)])[:,1].reshape(raw.shape)
    cost=((1-motion_weight)*raw+motion_weight*residual)/5-probability-.08*(ascore[:,None]+bscore[None]);valid=raw<=CFG['linkGateUm']
    if len(b)>CFG['candidateK']:
        near=np.argpartition(np.where(valid,cost,np.inf),CFG['candidateK']-1,axis=1)[:,:CFG['candidateK']];sparse=np.zeros_like(valid);sparse[np.arange(len(a))[:,None],near]=True;valid&=sparse
    rows,cols=linear_sum_assignment(np.where(valid,cost,1e6)); accepted=[(i,j) for i,j in zip(rows,cols) if valid[i,j]]
    links=[(int(aid[i]),int(bid[j]),float(cost[i,j])) for i,j in accepted];used={j for _,j in accepted}
    if threshold<=1:
        for i in np.flatnonzero(division>=threshold):
            choices=[j for j in np.argsort(cost[i]) if valid[i,j] and j not in used and cost[i,j]<0]
            if choices: j=choices[0];links.append((int(aid[i]),int(bid[j]),float(cost[i,j])));used.add(j)
    return links

def build_graph(maps,estimate,fusion_weight,count_multiplier,division_threshold,motion_weight):
    pools=[extract_pool(*item,fusion_weight) for item in maps];counts=allocate_counts([len(p[0]) for p in pools],estimate*count_multiplier)
    nodes=[];edges=[];previous=None;next_id=1
    for t,(pool,count) in enumerate(zip(pools,counts)):
        coords,scores,motion,division=[x[:count] for x in pool];ids=np.arange(next_id,next_id+count);next_id+=count
        nodes.extend((int(i),t,float(z),float(y),float(x),float(s)) for i,(z,y,x),s in zip(ids,coords,scores))
        if previous is not None:edges.extend(link_pair(previous[0],previous[1],coords,ids,motion,previous[3],scores,previous[4],division_threshold,motion_weight))
        previous=(coords,ids,motion,scores,division)
    return nodes,edges


## 9. Held-out scoring and model selection


In [ ]:
def score_edges(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_total):
    pred_df = pd.DataFrame(pred_nodes, columns=['node_id','t','z','y','x','score'])
    mapping = {}
    for t, gt_part in gt_nodes.groupby('t'):
        pred_part = pred_df[pred_df.t == t]
        if pred_part.empty or gt_part.empty: continue
        pred_xyz = pred_part[['z','y','x']].to_numpy(float)
        gt_xyz = gt_part[['z','y','x']].to_numpy(float)
        distances = np.linalg.norm((pred_xyz[:,None,:]-gt_xyz[None,:,:])*SPACING_UM, axis=2)
        rows, cols = linear_sum_assignment(distances)
        pred_ids = pred_part.node_id.to_numpy(); gt_ids = gt_part.node_id.to_numpy()
        for i,j in zip(rows,cols):
            if distances[i,j] <= MATCH_RADIUS_UM: mapping[int(pred_ids[i])] = int(gt_ids[j])

    gt_edge_set = {tuple(map(int,e)) for e in gt_edges}
    gt_out = {s for s,_ in gt_edge_set}; gt_in = {t for _,t in gt_edge_set}
    tp_pairs, fp = set(), 0
    for source,target,_ in pred_edges:
        ms, mt = mapping.get(source), mapping.get(target)
        pair = (ms,mt)
        if ms is not None and mt is not None and pair in gt_edge_set:
            tp_pairs.add(pair)
        elif (ms is not None and ms in gt_out) or (mt is not None and mt in gt_in):
            fp += 1
    tp = len(tp_pairs); fn = len(gt_edge_set - tp_pairs)
    jaccard = tp / max(tp+fp+fn, 1)
    ratio = (len(pred_nodes)-estimated_total)/estimated_total if np.isfinite(estimated_total) and estimated_total>0 else np.nan
    adjusted = max(0, jaccard*(1-0.1*ratio)) if np.isfinite(ratio) else jaccard
    return {'edge_tp':tp,'edge_fp':fp,'edge_fn':fn,'edge_jaccard':jaccard,'node_count':len(pred_nodes),
            'node_ratio_delta':ratio,'adjusted_edge_jaccard':adjusted,'matched_nodes':len(set(mapping.values()))}


In [ ]:
def score_divisions_local(pred_nodes,pred_edges,gt_nodes,gt_edges):
    """Conservative local fork score used for selection; official scorer remains authoritative."""
    pred=pd.DataFrame(pred_nodes,columns=['node_id','t','z','y','x','score']); mapping={}
    for t,g in gt_nodes.groupby('t'):
        p=pred[pred.t==t]
        if p.empty:continue
        distance=np.linalg.norm((p[['z','y','x']].to_numpy()[:,None]-g[['z','y','x']].to_numpy()[None])*SPACING_UM,axis=2)
        rows,cols=linear_sum_assignment(distance)
        for i,j in zip(rows,cols):
            if distance[i,j]<=MATCH_RADIUS_UM:mapping[int(p.iloc[i].node_id)]=int(g.iloc[j].node_id)
    gt_out=defaultdict(list);pred_out=defaultdict(list)
    for a,b in gt_edges:gt_out[int(a)].append(int(b))
    for a,b,_ in pred_edges:pred_out[int(a)].append(int(b))
    gt_forks={a for a,v in gt_out.items() if len(v)>=2};pred_forks={a for a,v in pred_out.items() if len(v)>=2}
    recovered={mapping[a] for a in pred_forks if a in mapping and mapping[a] in gt_forks}
    tp=len(recovered);fp=sum(1 for a in pred_forks if a in mapping and mapping[a] not in gt_forks);fn=len(gt_forks-recovered)
    return tp,fp,fn

validation_rows=[];map_cache={}
for path in validation_movies:
    print('validation inference:',path.stem);maps=infer_movie(path);map_cache[path]=maps;gt_nodes,gt_edges,estimate=load_geff(path.with_suffix('.geff'))
    for fusion,count,division,motion_weight in product(CFG['fusionWeights'],CFG['countMultipliers'],CFG['divisionThresholds'],CFG['motionWeights']):
        nodes,edges=build_graph(maps,estimate,fusion,count,division,motion_weight);metrics=score_edges(nodes,edges,gt_nodes,gt_edges,estimate);dtp,dfp,dfn=score_divisions_local(nodes,edges,gt_nodes,gt_edges)
        validation_rows.append({'dataset':path.stem,'fusionWeight':fusion,'countMultiplier':count,'divisionThreshold':division,'motionWeight':motion_weight,'division_tp':dtp,'division_fp':dfp,'division_fn':dfn,**metrics})
validation=pd.DataFrame(validation_rows);validation['weight']=validation.edge_tp+validation.edge_fp+validation.edge_fn
summary=[]
for key,g in validation.groupby(['fusionWeight','countMultiplier','divisionThreshold','motionWeight']):
    tp,fp,fn=g[['edge_tp','edge_fp','edge_fn']].sum();dtp,dfp,dfn=g[['division_tp','division_fp','division_fn']].sum();adjusted=np.average(g.adjusted_edge_jaccard,weights=g.weight);dj=dtp/max(dtp+dfp+dfn,1)
    summary.append({'fusionWeight':key[0],'countMultiplier':key[1],'divisionThreshold':key[2],'motionWeight':key[3],'adjustedEdgeJaccard':adjusted,'edgeJaccard':tp/max(tp+fp+fn,1),'divisionJaccard':dj,'combinedScore':adjusted+.1*dj})
summary=pd.DataFrame(summary).sort_values('combinedScore',ascending=False);BEST=summary.iloc[0].to_dict()
display(summary);print('selected:',BEST);validation.to_csv('/kaggle/working/dayFourValidation.csv',index=False);summary.to_csv('/kaggle/working/dayFourValidationSummary.csv',index=False)


## 10. Frozen test inference

The hidden node total is unavailable. The notebook transfers only the median nodes-per-frame estimate from training movies of the same embryo prefix. No test label information is used.


In [ ]:
def node_rate_prior(paths):
    rates=defaultdict(list)
    for path in paths:
        _,_,estimate=load_geff(path.with_suffix('.geff'))
        if np.isfinite(estimate):rates[embryo_id(path)].append(estimate/len(TimeChunkedZarr(path)))
    pooled=[x for values in rates.values() for x in values]
    result={k:float(np.median(v)) for k,v in rates.items()};result['global']=float(np.median(pooled));return result

RATES=node_rate_prior(fit_movies)
COLUMNS=['dataset','row_type','node_id','t','z','y','x','source_id','target_id'];parts=[];run_rows=[]
for path in test_movies:
    started=time.time();print('test inference:',path.stem);maps=infer_movie(path)
    estimate=RATES.get(embryo_id(path),RATES['global'])*len(maps)
    nodes,edges=build_graph(maps,estimate,BEST['fusionWeight'],BEST['countMultiplier'],BEST['divisionThreshold'],BEST['motionWeight'])
    node_rows=[(path.stem,'node',i,t,round(z),round(y),round(x),-1,-1) for i,t,z,y,x,s in nodes]
    edge_rows=[(path.stem,'edge',-1,-1,-1,-1,-1,a,b) for a,b,c in edges]
    parts.append(pd.DataFrame(node_rows+edge_rows,columns=COLUMNS));run_rows.append({'dataset':path.stem,'estimatedNodes':estimate,'nodes':len(nodes),'edges':len(edges),'seconds':time.time()-started})
submission=pd.concat(parts,ignore_index=True);submission.insert(0,'id',np.arange(len(submission),dtype=np.int64))
for column in ['id','node_id','t','z','y','x','source_id','target_id']:submission[column]=submission[column].astype(np.int64)
run_summary=pd.DataFrame(run_rows);display(run_summary)


In [ ]:
def validate_submission(df,expected):
    assert list(df.columns)==['id']+COLUMNS and df.id.is_unique and set(df.dataset)==set(expected)
    for dataset,part in df.groupby('dataset'):
        nodes=part[part.row_type=='node'];edges=part[part.row_type=='edge'];times=dict(zip(nodes.node_id,nodes.t));ids=set(times)
        assert nodes.node_id.is_unique and set(edges.source_id)<=ids and set(edges.target_id)<=ids
        assert all(times[b]==times[a]+1 for a,b in zip(edges.source_id,edges.target_id))
        if len(edges):assert edges.target_id.value_counts().max()<=1 and edges.source_id.value_counts().max()<=2
    return True

assert validate_submission(submission,[p.stem for p in test_movies])
submission.to_csv('/kaggle/working/submission.csv',index=False);run_summary.to_csv('/kaggle/working/runSummary.csv',index=False)
with open('/kaggle/working/selectedConfiguration.json','w') as f:json.dump(BEST,f,indent=2)
print(f'wrote {len(submission):,} rows to /kaggle/working/submission.csv')


## Outputs

- `submission.csv`: competition submission;
- `dayFourModel.pt`: trained network weights;
- `trainingHistory.csv`: optimization trace;
- `dayFourValidation.csv` and `dayFourValidationSummary.csv`: selection evidence;
- `selectedConfiguration.json`: frozen inference choices;
- `runSummary.csv`: dataset-level counts and runtime.

The network maps are cached within each movie, so the validation sweep does not repeat neural inference. GPU acceleration is strongly recommended.
